# CATS-GSM on real Porto taxi trajectories (trace-driven evaluation)

**What this notebook does:** replays real taxi routes (ECML/PKDD 2015 Porto dataset, CC BY 4.0) through the CATS-GSM simulator and the baseline schemes, then prints the tables used in the paper.

**Real:** vehicle paths and speeds (15 s GPS samples, interpolated to 1 Hz). **Modelled:** IMU, GPS noise/jamming/spoofing, serving cell and timing advance (synthetic base-station layout), energy.

**Steps:** (1) Settings -> Internet **On** (or attach the Porto dataset as input). (2) Run all. (3) Download `results_real.zip` from the output panel and send it to Claude.

Code: https://github.com/kapopat1980/cats-gsm (original synthetic study).

In [ ]:
# ---- settings ----
QUICK = False          # True = tiny test run (~1 min) to check everything works
import os
if QUICK:
    os.environ.update(N_DRIVE='10', N_ARMED='10', N_FA='3')
else:
    os.environ.update(N_DRIVE='100', N_ARMED='150', N_FA='40')
# optional: limit rows read from train.csv (None = all 1.7 M trips)
# os.environ['NROWS'] = '400000'
print({k: os.environ[k] for k in ('N_DRIVE','N_ARMED','N_FA')})

In [ ]:
# ---- locate or download the Porto trips ----
import glob, os, zipfile, urllib.request
found = [p for p in glob.glob('/kaggle/input/**/*.csv', recursive=True) if 'train' in os.path.basename(p).lower()]
if found:
    csv = max(found, key=os.path.getsize); print('using attached data:', csv)
else:
    url = 'https://archive.ics.uci.edu/static/public/339/taxi+service+trajectory+prediction+challenge+ecml+pkdd+2015.zip'
    print('downloading (509 MB) from UCI ... needs Internet = On'); urllib.request.urlretrieve(url, 'porto.zip')
    zipfile.ZipFile('porto.zip').extract('train.csv.zip', 'porto'); zipfile.ZipFile('porto/train.csv.zip').extractall('porto')
    csv = 'porto/train.csv'; print('ready:', csv, round(os.path.getsize(csv)/1e6), 'MB')
os.environ['PORTO_CSV'] = csv

In [ ]:
%%writefile world.py
"""Synthetic urban GSM world and trace generator for the CATS-GSM study.

All quantities are SI (metres, seconds). Everything here is *model assumption*,
documented in the manuscript (Section 5) -- no field data are used.
"""
import numpy as np

AREA = 12000.0          # city side [m]
SPACING = 300.0         # road grid spacing [m]
NODES = int(AREA / SPACING)
TA_STEP = 553.5         # GSM timing-advance step [m]
ISD = 1000.0            # nominal inter-site distance [m]

SIG_GPS = 4.0           # open-sky GPS position noise [m]
SIG_GPS_CANYON = 20.0   # urban-canyon GPS noise [m]
P_FIX_CANYON = 0.45     # per-second fix availability inside canyon zones
SIG_ACC = 0.15          # accelerometer noise [m/s^2]
SIG_GYR = 0.01          # gyro noise [rad/s]


def make_world(seed=12345, area=AREA):
    rng = np.random.default_rng(seed)
    pts = []
    dy = ISD * np.sqrt(3) / 2
    y, row = -2 * ISD, 0
    while y < area + 2 * ISD:
        x = -2 * ISD + (ISD / 2 if row % 2 else 0.0)
        while x < area + 2 * ISD:
            pts.append((x + rng.uniform(-150, 150), y + rng.uniform(-150, 150)))
            x += ISD
        y += dy
        row += 1
    bts = np.array(pts)
    # urban-canyon zones (unknown to the algorithms): ~15 % of the area
    nz = max(40, int(40 * (area / AREA) ** 2))
    zc = rng.uniform(0, area, size=(nz, 2))
    zr = rng.uniform(250, 500, size=nz)
    return dict(bts=bts, zc=zc, zr=zr)


WORLD = make_world()


def set_world(area, seed=12345):
    """Rebuild the base-station layout / canyon zones for a larger area (mutates WORLD in place)."""
    WORLD.clear(); WORLD.update(make_world(seed, area))


def in_canyon(x, y):
    d = np.hypot(WORLD['zc'][:, 0] - x, WORLD['zc'][:, 1] - y)
    return bool(np.any(d < WORLD['zr']))


# ----------------------------------------------------------------- kinematics
DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]


def drive(rng, start_node, n_steps, p_stop=0.25):
    """Random-walk driving on the grid. Returns arrays x,y,v,th,a,w (length n_steps)."""
    i, j = start_node
    d = int(rng.integers(0, 4))
    S = SPACING
    x = np.zeros(n_steps); y = np.zeros(n_steps); v = np.zeros(n_steps)
    th = np.zeros(n_steps); a = np.zeros(n_steps); w = np.zeros(n_steps)
    t = 0
    cx, cy = i * S, j * S
    cur_v = 0.0
    cur_th = d * np.pi / 2

    def target(i, j, d):
        dx, dy = DIRS[d]
        arterial = (j % 4 == 0) if dx != 0 else (i % 4 == 0)
        return 18.0 if arterial else 10.0

    def choose_dir(i, j, d):
        r = rng.random()
        nd = d if r < 0.6 else (d + 1) % 4 if r < 0.8 else (d + 3) % 4
        ni, nj = i + DIRS[nd][0], j + DIRS[nd][1]
        if not (0 <= ni <= NODES and 0 <= nj <= NODES):
            for k in (1, 3, 2, 0):
                nd = (d + k) % 4
                ni, nj = i + DIRS[nd][0], j + DIRS[nd][1]
                if 0 <= ni <= NODES and 0 <= nj <= NODES:
                    break
        return nd

    d = choose_dir(i, j, d)
    while t < n_steps:
        # start of a segment from node (i,j) in direction d
        dx, dy = DIRS[d]
        vt = target(i, j, d)
        nd = choose_dir(i + dx, j + dy, d)
        stop = rng.random() < p_stop
        turn = nd != d
        v_end = 0.0 if stop else (4.5 if turn else target(i + dx, j + dy, nd) * 0.9)
        tgt_th = d * np.pi / 2
        s = 0.0
        while s < S and t < n_steps:
            rem = S - s
            v_allow = np.sqrt(v_end ** 2 + 2 * 2.5 * max(rem, 0.0))
            v_des = min(vt, v_allow)
            nv = cur_v + np.clip(v_des - cur_v, -2.5, 1.5)
            nv = max(nv, 0.0)
            if nv < 0.5 and v_des > 1.0:
                nv = 0.5
            s += 0.5 * (cur_v + nv)
            # heading rotates towards segment heading over ~2 s
            dth = (tgt_th - cur_th + np.pi) % (2 * np.pi) - np.pi
            rot = np.clip(dth, -np.pi / 4, np.pi / 4)
            a[t] = nv - cur_v
            w[t] = rot
            cur_th += rot
            cur_v = nv
            x[t] = cx + dx * min(s, S) + 0.0
            y[t] = cy + dy * min(s, S)
            v[t] = cur_v
            th[t] = cur_th
            t += 1
        cx += dx * S; cy += dy * S; i += dx; j += dy
        if stop:
            dur = int(rng.integers(8, 40))
            for _ in range(dur):
                if t >= n_steps:
                    break
                x[t] = cx; y[t] = cy; v[t] = 0.0; th[t] = cur_th; a[t] = -cur_v if _ == 0 else 0
                cur_v = 0.0
                t += 1
        d = nd
    return x, y, v, th, a, w


# ----------------------------------------------------------------- radio
def cell_series(rng, x, y, v, dt=1.0):
    """Serving cell and timing advance along a trajectory.
    RSSI_i = -30 - 35 log10(d_i) + s_spatial_i + s_temporal_i ; idle reselection with 3 dB hysteresis."""
    bts = WORLD['bts']
    nb = len(bts)
    n = len(x)
    s_sp = rng.normal(0, 5.0, nb)
    s_tm = rng.normal(0, 2.5, nb)
    serving = np.zeros(n, dtype=np.int32)
    ta = np.zeros(n, dtype=np.int16)
    cur = -1
    px, py = x[0], y[0]
    for t in range(n):
        step = np.hypot(x[t] - px, y[t] - py)
        px, py = x[t], y[t]
        rho_s = np.exp(-step / 100.0)
        rho_t = np.exp(-dt / 120.0)
        s_sp = rho_s * s_sp + np.sqrt(1 - rho_s ** 2) * 5.0 * rng.standard_normal(nb)
        s_tm = rho_t * s_tm + np.sqrt(1 - rho_t ** 2) * 2.5 * rng.standard_normal(nb)
        d = np.hypot(bts[:, 0] - x[t], bts[:, 1] - y[t])
        rssi = -30.0 - 35.0 * np.log10(np.maximum(d, 10.0)) + s_sp + s_tm
        best = int(np.argmax(rssi))
        if cur < 0 or (best != cur and rssi[best] > rssi[cur] + 4.0):
            cur = best
        serving[t] = cur
        dc = d[cur]
        ta[t] = max(0, int(round(dc / TA_STEP + rng.normal(0, 0.35))))
    return serving, ta


# ----------------------------------------------------------------- traces
def _sensors(rng, x, y, v, th, a, w, dt=1.0):
    n = len(x)
    ba = rng.normal(0, 0.04)
    bw = rng.normal(0, 0.003)
    imu_a = a + ba + rng.normal(0, SIG_ACC, n)
    imu_w = w + bw + rng.normal(0, SIG_GYR, n)
    # GPS availability / noise
    gps_ok = np.ones(n, dtype=bool)
    sig = np.full(n, SIG_GPS)
    for t in range(n):
        if in_canyon(x[t], y[t]):
            gps_ok[t] = rng.random() < P_FIX_CANYON
            sig[t] = SIG_GPS_CANYON
    gx = x + rng.normal(0, 1, n) * sig
    gy = y + rng.normal(0, 1, n) * sig
    gv = np.maximum(v + rng.normal(0, 0.15, n), 0.0)
    gth = th + rng.normal(0, 1, n) * (0.04 + 0.5 / np.maximum(v, 0.3))
    return dict(imu_a=imu_a, imu_w=imu_w, gps_ok=gps_ok, gx=gx, gy=gy, gv=gv, gth=gth, ba=ba, bw=bw)


def driving_trace(seed, n=3600, route=None):
    rng = np.random.default_rng(seed)
    if route is not None:          # real trajectory (x, y, v, th, a, w) at 1 Hz
        x, y, v, th, a, w = [np.asarray(r, float) for r in route]; n = len(x)
    else:
        start = (int(rng.integers(5, NODES - 5)), int(rng.integers(5, NODES - 5)))
        x, y, v, th, a, w = drive(rng, start, n)
    sv, ta = cell_series(rng, x, y, v)
    tr = dict(x=x, y=y, v=v, th=th, a=a, w=w, serving=sv, ta=ta)
    tr.update(_sensors(rng, x, y, v, th, a, w))
    tr['n'] = n
    return tr


ACC_FA_RATE = 0.25 / 3600.0   # spurious motion-sensor triggers per second
ACC_PD = 0.90                 # prob. the motion sensor detects theft motion onset


def armed_trace(seed, n_pre, n_theft, attack, theft=True, acc_pd=ACC_PD, route=None, park=None):
    """Parked & armed for n_pre s (random location), then (optionally) theft driving for n_theft s."""
    rng = np.random.default_rng(seed)
    i = int(rng.integers(5, NODES - 5)); j = int(rng.integers(5, NODES - 5))
    px, py = i * SPACING, j * SPACING
    if park is not None: px, py = park
    elif route is not None: px, py = float(route[0][0]), float(route[1][0])
    if route is not None and theft: n_theft = len(route[0])
    n = n_pre + (n_theft if theft else 0)
    x = np.full(n, px); y = np.full(n, py); v = np.zeros(n); th = np.zeros(n)
    a = np.zeros(n); w = np.zeros(n)
    if theft:
        if route is not None:
            xs, ys, vs, ths, as_, ws = [np.asarray(r, float) for r in route]
            xs = xs - xs[0] + px; ys = ys - ys[0] + py
        else:
            xs, ys, vs, ths, as_, ws = drive(rng, (i, j), n_theft, p_stop=0.15)
        x[n_pre:] = xs; y[n_pre:] = ys; v[n_pre:] = vs; th[n_pre:] = ths
        a[n_pre:] = as_; w[n_pre:] = ws
    sv, ta = cell_series(rng, x, y, v)
    tr = dict(x=x, y=y, v=v, th=th, a=a, w=w, serving=sv, ta=ta)
    tr.update(_sensors(rng, x, y, v, th, a, w))
    t0 = n_pre if theft else -1
    gx, gy, gok = tr['gx'].copy(), tr['gy'].copy(), tr['gps_ok'].copy()
    if theft:
        if attack == 'jam':
            gok[t0 + int(rng.integers(0, 30)):] = False
        elif attack == 'replay':      # receiver spoofed to keep reporting the parked position
            gx[t0:] = px + rng.normal(0, SIG_GPS, n - t0)
            gy[t0:] = py + rng.normal(0, SIG_GPS, n - t0)
            tr['gv'] = tr['gv'].copy(); tr['gth'] = tr['gth'].copy()
            tr['gv'][t0:] = np.abs(rng.normal(0, 0.15, n - t0))     # spoofer also feeds a stationary velocity
            tr['gth'][t0:] = rng.uniform(-np.pi, np.pi, n - t0)
        elif attack == 'drift':       # slow carry-off: offset grows 2 m/s opposite to a random bearing
            ang = rng.uniform(0, 2 * np.pi)
            off = 2.0 * np.arange(n - t0)
            gx[t0:] += off * np.cos(ang)
            gy[t0:] += off * np.sin(ang)
    tr['gx'], tr['gy'], tr['gps_ok'] = gx, gy, gok
    # motion-sensor events
    fa_times = np.nonzero(rng.random(n) < ACC_FA_RATE)[0]
    tr['acc_fa'] = fa_times[fa_times < (t0 if theft else n)]
    tr['acc_hit'] = (t0 + int(rng.integers(1, 5))) if (theft and rng.random() < acc_pd) else -1
    tr['t0'] = t0
    tr['n'] = n
    tr['attack'] = attack
    tr['park'] = (px, py)
    return tr

In [ ]:
%%writefile algos.py
"""Baselines and the proposed CATS-GSM algorithm.  All algorithms consume pre-generated traces
(see world.py) and therefore see *identical* noise realisations."""
import numpy as np
from world import WORLD, TA_STEP

# ------------------------------------------------------------------ energy model (assumptions)
E_FIX = 0.60       # J, one GPS acquisition (hot start, ~5 s at 120 mW)
P_GPS_ON = 0.12    # W, receiver kept on between closely spaced attempts
E_ATT = 1.80       # J, GPRS attach / PDP-context set-up
E_MSG = 0.50       # J, one small GPRS packet inside an open context
CTX_T = 15.0       # s, context idle timeout
E_SMS = 1.20       # J, one SMS
P_BASE = 0.010     # W, MCU + camped modem (common to all schemes)
P_IMU = 0.001      # W, 1 Hz accelerometer+gyro sampling (CATS)
P_CELLQ = 0.002    # W, 1 Hz serving-cell/TA query (CATS, cell-only)
P_ACC = 0.0002     # W, wake-on-motion accelerometer (reactive baselines)
BATT_J = 3.7 * 2.0 * 3600.0   # 2000 mAh @ 3.7 V


class Meter:
    def __init__(self):
        self.e_gps = 0.0; self.e_tx = 0.0; self.n_fix = 0; self.n_fail = 0; self.n_tx = 0
        self.last_gps = -1e9; self.ctx_until = -1e9

    def fix(self, t, ok):
        cost = min(E_FIX, P_GPS_ON * (t - self.last_gps))
        self.e_gps += cost; self.last_gps = t
        if ok: self.n_fix += 1
        else: self.n_fail += 1
        return ok

    def tx(self, t, sms=False):
        if sms:
            self.e_tx += E_SMS
        else:
            self.e_tx += E_MSG + (E_ATT if t > self.ctx_until else 0.0)
            self.ctx_until = t + CTX_T
        self.n_tx += 1

    def total(self, dur, extra_power=0.0):
        return self.e_gps + self.e_tx + (P_BASE + extra_power) * dur


# ------------------------------------------------------------------ server estimators
def est_from_reports(reports, n, mode):
    """reports: list of (t,x,y,v,th). mode 'zoh' or 'cv'. Returns est arrays and first-report index."""
    ex = np.full(n, np.nan); ey = np.full(n, np.nan)
    if not reports:
        return ex, ey
    k = 0
    for t in range(n):
        while k + 1 < len(reports) and reports[k + 1][0] <= t:
            k += 1
        r = reports[k]
        if r[0] > t:
            continue
        if mode == 'zoh':
            ex[t], ey[t] = r[1], r[2]
        else:
            dt = t - r[0]
            ex[t] = r[1] + r[3] * np.cos(r[4]) * dt
            ey[t] = r[2] + r[3] * np.sin(r[4]) * dt
    return ex, ey


def track_error(tr, ex, ey, lo=0, hi=None):
    hi = tr['n'] if hi is None else hi
    d = np.hypot(ex[lo:hi] - tr['x'][lo:hi], ey[lo:hi] - tr['y'][lo:hi])
    d = d[~np.isnan(d)]
    return d


def pack(tr, m, ex, ey, extra_power):
    err = track_error(tr, ex, ey)
    dur = tr['n']
    E = m.total(dur, extra_power)
    return dict(mean=float(err.mean()), rmse=float(np.sqrt((err ** 2).mean())), p95=float(np.percentile(err, 95)),
                energy_J=float(E), power_mW=float(E / dur * 1e3), msgs=m.n_tx, fixes=m.n_fix + m.n_fail,
                life_days=float(BATT_J / (E / dur) / 86400.0))


# ------------------------------------------------------------------ D1 periodic GPS + report
def run_periodic(tr, T):
    n = tr['n']; m = Meter(); rep = []; nxt = 0
    for t in range(n):
        if t >= nxt:
            if m.fix(t, tr['gps_ok'][t]):
                rep.append((t, tr['gx'][t], tr['gy'][t], tr['gv'][t], tr['gth'][t])); m.tx(t); nxt = t + T
            else:
                nxt = t + 1
    ex, ey = est_from_reports(rep, n, 'zoh')
    return pack(tr, m, ex, ey, 0.0)


# ------------------------------------------------------------------ D2 distance-based reporting
def run_distance(tr, D, T=5):
    n = tr['n']; m = Meter(); rep = []; nxt = 0; last = None
    for t in range(n):
        if t >= nxt:
            if m.fix(t, tr['gps_ok'][t]):
                gx, gy = tr['gx'][t], tr['gy'][t]
                if last is None or np.hypot(gx - last[0], gy - last[1]) >= D:
                    rep.append((t, gx, gy, tr['gv'][t], tr['gth'][t])); m.tx(t); last = (gx, gy)
                nxt = t + T
            else:
                nxt = t + 1
    ex, ey = est_from_reports(rep, n, 'zoh')
    return pack(tr, m, ex, ey, 0.0)


# ------------------------------------------------------------------ D3 dead-reckoning, speed-adaptive sampling
def run_dr_threshold(tr, theta, kappa=0.3, tmax=60.0):
    n = tr['n']; m = Meter(); rep = []; nxt = 0
    for t in range(n):
        if t >= nxt:
            if m.fix(t, tr['gps_ok'][t]):
                gx, gy, gv, gth = tr['gx'][t], tr['gy'][t], tr['gv'][t], tr['gth'][t]
                send = True
                if rep:
                    r = rep[-1]; dt = t - r[0]
                    px = r[1] + r[3] * np.cos(r[4]) * dt; py = r[2] + r[3] * np.sin(r[4]) * dt
                    send = np.hypot(px - gx, py - gy) > theta
                if send:
                    rep.append((t, gx, gy, gv, gth)); m.tx(t)
                nxt = t + min(max(theta / (kappa * max(gv, 0.5)), 1.0), tmax)
            else:
                nxt = t + 1
    ex, ey = est_from_reports(rep, n, 'cv')
    return pack(tr, m, ex, ey, 0.0)


# ------------------------------------------------------------------ D4 cell-ID only
def run_cellid(tr):
    n = tr['n']; m = Meter(); bts = WORLD['bts']
    ex = np.zeros(n); ey = np.zeros(n); cur = tr['serving'][0]; m.tx(0)
    for t in range(n):
        if tr['serving'][t] != cur:
            cur = tr['serving'][t]; m.tx(t)
        ex[t], ey[t] = bts[cur]
    return pack(tr, m, ex, ey, P_CELLQ)


# ------------------------------------------------------------------ CATS driving mode
def _wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


def run_cats_drive(tr, eps, use_imu=True, use_trig=True, sched='learn'):
    n = tr['n']; m = Meter(); rep = []
    have = False; ex = ey = ev = eth = 0.0; ba = bw = 0.0; chat = 0.02
    tl = 0; nxt = 0; stationary = False; yaw_acc = 0.0; srv = None; last_rep_t = -99
    ia, iw = tr['imu_a'], tr['imu_w']
    for t in range(n):
        a_c = 0.0
        if have:
            if use_imu:
                a_c = ia[t] - ba; w_c = iw[t] - bw
                ev = max(0.0, ev + a_c); eth += w_c; yaw_acc += abs(w_c)
            ex += ev * np.cos(eth); ey += ev * np.sin(eth)
        do_fix = t >= nxt
        if have and use_imu and use_trig:
            if stationary and a_c > 0.4:
                do_fix = True
            if yaw_acc > 0.5 and nxt > t + 2:
                nxt = t + 2
        if do_fix:
            if m.fix(t, tr['gps_ok'][t]):
                gx, gy, gv, gth = tr['gx'][t], tr['gy'][t], tr['gv'][t], tr['gth'][t]
                if not have:
                    ex, ey, ev, eth = gx, gy, gv, gth; have = True
                else:
                    dt = max(t - tl, 1)
                    err = np.hypot(ex - gx, ey - gy)
                    chat = max(0.7 * chat + 0.3 * max(err - 8.0, 0.0) / dt ** 1.5, 1e-4)
                    if use_imu:
                        ba += 0.3 * np.clip((ev - gv) / dt, -0.1, 0.1)
                        if gv > 2 and ev > 2:
                            bw += 0.3 * np.clip(_wrap(eth - gth) / dt, -0.01, 0.01)
                    ex, ey = gx, gy
                    ev = (0.3 * ev + 0.7 * gv) if use_imu else gv
                    if gv > 2:
                        eth = gth
                stationary = gv < 0.3; yaw_acc = 0.0; tl = t
                if sched == 'learn':
                    delta = min(max((eps / chat) ** (1 / 1.5), 2.0), 300.0)
                else:
                    delta = min(max(eps / (0.3 * max(ev, 0.5)), 2.0), 60.0)
                if stationary: delta = min(delta, 60.0)
                nxt = t + delta
                send = srv is None
                if srv is not None:
                    dtp = t - srv[0]
                    px = srv[1] + srv[3] * np.cos(srv[4]) * dtp; py = srv[2] + srv[3] * np.sin(srv[4]) * dtp
                    send = np.hypot(px - gx, py - gy) > 0.7 * eps
                if send:
                    srv = (t, gx, gy, ev, eth); rep.append(srv); m.tx(t); last_rep_t = t
            else:
                nxt = t + 1
        elif have and use_imu and t - last_rep_t >= 2:
            dtp = t - srv[0]
            px = srv[1] + srv[3] * np.cos(srv[4]) * dtp; py = srv[2] + srv[3] * np.sin(srv[4]) * dtp
            if np.hypot(px - ex, py - ey) > eps:
                srv = (t, ex, ey, ev, eth); rep.append(srv); m.tx(t); last_rep_t = t
    ex_, ey_ = est_from_reports(rep, n, 'cv')
    return pack(tr, m, ex_, ey_, P_IMU if use_imu else 0.0)


# =====================================================================================
#                                     SECURITY (armed mode)
# =====================================================================================
class SecParams:
    verify_window = 120      # s
    confirm_dist = 100.0     # m, displacement that confirms theft
    cats_interval = 5        # s
    base_interval = 10       # s
    failsafe = 30            # s, baselines: no fix after trigger -> alarm
    cats_jam_wait = 20       # s
    imu_v_thr = 5.0          # m/s integrated speed that counts as IMU-motion
    chi_win = 8              # fixes in range-residual window
    chi_thr = 3.0            # mean z^2 threshold
    chi_sigma = 250.0        # m, TA range residual sigma
    ring_sigma_k = 2.5       # (|d_gps - TA*step|/step) threshold
    knear = 6


def _nearest_rank(px, py, cell):
    bts = WORLD['bts']
    d = np.hypot(bts[:, 0] - px, bts[:, 1] - py)
    return int(np.sum(d < d[cell]))     # 0 = nearest


def cell_novelty_events(tr, t_end=None):
    """Times at which the serving cell is new w.r.t. the 'home set' learnt in the first 300 s of arming."""
    sv = tr['serving']; n = tr['n'] if t_end is None else t_end
    H = set(int(c) for c in sv[:300])
    ev = []
    ch = np.nonzero(np.diff(sv[:n]) != 0)[0] + 1
    for t in ch:
        c = int(sv[t])
        if t >= 300 and c not in H:
            ev.append(int(t)); H.add(c)
    return ev


def cell_triggers(tr, t_end=None):
    """Cell-sentinel candidate triggers: 2 novel cells within 180 s."""
    ev = cell_novelty_events(tr, t_end)
    out = []
    for i in range(1, len(ev)):
        if ev[i] - ev[i - 1] <= 180:
            out.append(ev[i])
    return out, ev


def _verify(tr, tau, mode, m, P):
    """Verification window after a trigger at tau. Returns (alarm_time or None, reason)."""
    n = tr['n']; px, py = tr['park']; ok = tr['gps_ok']
    window = P.verify_window
    interval = P.cats_interval if mode.startswith('CATS') else P.base_interval
    ia = tr['imu_a']
    lo = max(0, tau - 300)
    ba_est = float(np.mean(ia[lo:max(lo + 1, tau - 5)])) if tau - lo > 10 else 0.0
    v_imu = 0.0; t_next = tau; any_ok = False; bad_imu = 0; bad_cell = 0
    last_ok = tau
    nov = cell_novelty_events(tr, min(n, tau + window + 1)) if mode in ('CATS', 'CATS-cell') else []
    for t in range(tau, min(n, tau + window + 1)):
        v_imu += ia[t] - ba_est
        if t >= t_next:
            success = m.fix(t, ok[t])
            if success:
                any_ok = True; last_ok = t; t_next = t + interval
                gx, gy, gv = tr['gx'][t], tr['gy'][t], tr['gv'][t]
                D = np.hypot(gx - px, gy - py)
                if mode in ('A3', 'CATS', 'CATS-imu'):
                    if abs(v_imu) > P.imu_v_thr and gv < 1.0:
                        bad_imu += 1
                    else:
                        bad_imu = 0
                    if bad_imu >= 2:
                        return t, 'spoof_imu'
                if mode in ('CATS', 'CATS-cell'):
                    c = tr['serving'][t]
                    bx, by = WORLD['bts'][c]
                    ring = abs(np.hypot(gx - bx, gy - by) - TA_STEP * tr['ta'][t]) / TA_STEP
                    rank = _nearest_rank(gx, gy, c)
                    if ring > P.ring_sigma_k or rank >= P.knear:
                        bad_cell += 1
                    else:
                        bad_cell = 0
                    if bad_cell >= 2:
                        return t, 'spoof_cell'
                if D > P.confirm_dist:
                    return t, 'theft'
            else:
                t_next = t + 1
                if mode in ('A2', 'A3') and (t - tau) >= P.failsafe and not any_ok:
                    return t, 'failsafe'
                if mode.startswith('CATS') and not any_ok and (t - tau) >= P.cats_jam_wait:
                    motion = (mode != 'CATS-cell' and abs(v_imu) > P.imu_v_thr) or any(tau - 60 <= e <= t for e in nov)
                    if motion:
                        return t, 'jam'
                if mode.startswith('CATS') and any_ok and (t - last_ok) >= P.cats_jam_wait:
                    motion = (mode != 'CATS-cell' and abs(v_imu) > P.imu_v_thr) or any(tau - 60 <= e <= t for e in nov)
                    if motion:
                        return t, 'jam'
    return None, 'cancel'


def armed_detect(tr, scheme, P=SecParams, rearm=False):
    """Run armed-mode logic. Returns dict(alarms=[(t,reason)], meter, extra_power, theft_alarm=(t,reason)|None)."""
    n = tr['n']; m = Meter(); alarms = []
    trig = list(tr['acc_fa'])
    if tr['acc_hit'] >= 0:
        trig.append(int(tr['acc_hit']))
    trig = sorted(trig)
    if scheme == 'A1':
        for t in trig:
            m.fix(t, True); m.tx(t, sms=True); alarms.append((t, 'acc'))
        return dict(alarms=alarms, meter=m, extra=P_ACC)
    triggers = [(t, 'acc') for t in trig]
    extra = P_ACC
    if scheme in ('CATS', 'CATS-cell'):
        ct, _ = cell_triggers(tr)
        triggers += [(t, 'cell') for t in ct]
        extra = P_IMU + P_CELLQ
    elif scheme == 'CATS-imu':
        extra = P_IMU
    triggers.sort()
    busy_until = -1
    for t, src in triggers:
        if t <= busy_until:
            continue
        at, reason = _verify(tr, t, scheme, m, P)
        busy_until = (at if at is not None else min(n, t + P.verify_window)) + 1
        if at is not None:
            alarms.append((at, reason)); m.tx(at, sms=True)
            if not rearm:
                break      # after an alarm the system leaves armed mode (owner is notified)
    return dict(alarms=alarms, meter=m, extra=extra)


# ------------------------------------------------------------------ post-alarm tracking
def ekf_track(meas, t_start, n, x0, y0, p0, qa=1.5):
    """Range-only EKF on TA rings. meas: list of (t, bx, by, ta)."""
    X = np.array([x0, y0, 0.0, 0.0]); Pm = np.diag([p0 ** 2, p0 ** 2, 8.0 ** 2, 8.0 ** 2])
    F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    G = np.array([[0.5, 0], [0, 0.5], [1, 0], [0, 1]], float)
    Q = G @ G.T * qa ** 2
    out = np.full((n, 2), np.nan); k = 0
    for t in range(t_start, n):
        X = F @ X; Pm = F @ Pm @ F.T + Q
        while k < len(meas) and meas[k][0] <= t:
            _, bx, by, ta = meas[k]; k += 1
            dx, dy = X[0] - bx, X[1] - by; r = max(np.hypot(dx, dy), 50.0)
            Hm = np.array([[dx / r, dy / r, 0, 0]])
            robs = max(ta * TA_STEP, 140.0); sig = 140.0 if ta == 0 else 250.0
            S = float((Hm @ Pm @ Hm.T)[0, 0]) + sig ** 2
            K = (Pm @ Hm.T / S).ravel()
            X = X + K * (robs - r)
            Pm = (np.eye(4) - np.outer(K, Hm.ravel())) @ Pm
        out[t] = X[:2]
    return out


def track_after_alarm(tr, scheme, alarm, P=SecParams, period=30):
    """Server-side track over the theft window for a scheme. Returns est arrays (n,) and the time GPS was distrusted."""
    n = tr['n']; t0 = tr['t0']; px, py = tr['park']; bts = WORLD['bts']
    ex = np.full(n, np.nan); ey = np.full(n, np.nan)
    ex[t0:] = px; ey[t0:] = py
    if alarm is None:
        return ex, ey, None
    ta_, reason = alarm
    untrusted_at = None
    meas = []
    t = ta_; zs = []
    trusted = True
    if scheme.startswith('CATS') and reason in ('jam', 'spoof_imu', 'spoof_cell'):
        untrusted_at = ta_
    last_gps_ok = ta_
    gps_est = (px, py)
    rep = []
    while t < n:
        ok = tr['gps_ok'][t]
        if scheme.startswith('CATS') and untrusted_at is not None:
            meas.append((t, bts[tr['serving'][t]][0], bts[tr['serving'][t]][1], int(tr['ta'][t])))
            t += period; continue
        if ok:
            last_gps_ok = t
            gx, gy = tr['gx'][t], tr['gy'][t]
            rep.append((t, gx, gy))
            if scheme.startswith('CATS'):
                c = tr['serving'][t]; bx, by = bts[c]
                z = (np.hypot(gx - bx, gy - by) - TA_STEP * tr['ta'][t]) / P.chi_sigma
                zs.append(z * z)
                if len(zs) >= P.chi_win and np.mean(zs[-P.chi_win:]) > P.chi_thr:
                    untrusted_at = t
                meas.append((t, bx, by, int(tr['ta'][t])))
            t += period
        else:
            if scheme.startswith('CATS') and t - last_gps_ok >= 60:
                untrusted_at = t
            t += 1 if (not scheme.startswith('CATS')) or untrusted_at is None else period
    # build estimate
    k = 0; cur = (px, py)
    ptr = 0
    for tt in range(ta_, n):
        while ptr < len(rep) and rep[ptr][0] <= tt and (untrusted_at is None or rep[ptr][0] < untrusted_at):
            cur = (rep[ptr][1], rep[ptr][2]); ptr += 1
        ex[tt], ey[tt] = cur
    if scheme.startswith('CATS') and untrusted_at is not None:
        p0 = min(18.0 * max(untrusted_at - t0, 1) / 2.0, 3000.0)
        m2 = [mm for mm in meas if mm[0] >= untrusted_at - 60] if reason == 'theft' else meas
        out = ekf_track(m2, untrusted_at, n, px, py, max(p0, 300.0))
        ex[untrusted_at:] = out[untrusted_at:, 0]; ey[untrusted_at:] = out[untrusted_at:, 1]
    return ex, ey, untrusted_at

In [ ]:
%%writefile exp.py
import sys, json, time, pickle
import numpy as np
from multiprocessing import Pool
import world, algos as A

DRIVE_SEEDS = list(range(1000, 1100))
ARMED_SEEDS = list(range(2000, 2150))
FA_SEEDS = list(range(9100, 9140))
ATTACKS = ['none', 'jam', 'replay', 'drift']
SCHEMES = ['A1', 'A2', 'A3', 'CATS-imu', 'CATS-cell', 'CATS']

EPS = (25, 35, 50, 70, 100, 140, 200, 280, 400, 600, 800)
DRIVE_CFG = ([('D1-periodic', 'T', T) for T in (10, 20, 30, 45, 60, 90, 120, 180, 300)] +
             [('D2-distance', 'D', D) for D in (100, 250, 500, 1000)] +
             [('D3-DR', 'theta', th) for th in EPS] +
             [('D3-IMU', 'eps', e) for e in EPS] +
             [('CATS', 'eps', e) for e in EPS] +
             [('CATS-noIMU', 'eps', e) for e in EPS] +
             [('D4-cellID', '-', 0)])


def run_cfg(tr, name, val):
    if name == 'D1-periodic': return A.run_periodic(tr, val)
    if name == 'D2-distance': return A.run_distance(tr, val)
    if name == 'D3-DR': return A.run_dr_threshold(tr, val)
    if name == 'D3-IMU': return A.run_cats_drive(tr, val, sched='speed')
    if name == 'CATS': return A.run_cats_drive(tr, val, sched='learn')
    if name == 'CATS-noIMU': return A.run_cats_drive(tr, val, use_imu=False)
    if name == 'D4-cellID': return A.run_cellid(tr)


def drive_job(seed):
    tr = world.driving_trace(seed)
    return {f'{nm}|{v}': run_cfg(tr, nm, v) for nm, _, v in DRIVE_CFG}


def armed_job(args):
    seed, atk, pd = args
    tr = world.armed_trace(seed, 900, 1200, atk, acc_pd=pd)
    out = {}
    d4 = None
    for sc in SCHEMES:
        d = A.armed_detect(tr, sc)
        th = [a for a in d['alarms'] if a[0] >= tr['t0']]
        alarm = th[0] if th else None
        ex, ey, untr = A.track_after_alarm(tr, sc, alarm)
        err = A.track_error(tr, ex, ey, tr['t0'], tr['n'])
        out[sc] = dict(delay=(alarm[0] - tr['t0']) if alarm else None, reason=alarm[1] if alarm else None,
                       mean=float(err.mean()), p95=float(np.percentile(err, 95)), untrusted=untr)
    bts = world.WORLD['bts']
    cx = bts[tr['serving'][tr['t0']:], 0]; cy = bts[tr['serving'][tr['t0']:], 1]
    e = np.hypot(cx - tr['x'][tr['t0']:], cy - tr['y'][tr['t0']:])
    out['D4-cellID'] = dict(delay=None, reason=None, mean=float(e.mean()), p95=float(np.percentile(e, 95)), untrusted=None)
    return out


def fa_job(seed):
    tr = world.armed_trace(seed, 86400, 0, 'none', theft=False)
    out = {}
    for sc in SCHEMES:
        d = A.armed_detect(tr, sc, rearm=True)
        out[sc] = dict(alarms=len(d['alarms']), energy_J=float(d['meter'].total(86400, d['extra'])),
                       fixes=d['meter'].n_fix + d['meter'].n_fail)
    return out


if __name__ == '__main__':
    t = time.time()
    res = {}
    with Pool(2) as p:
        res['drive'] = p.map(drive_job, DRIVE_SEEDS)
        print('drive done', time.time() - t, flush=True)
        jobs = [(s, atk, 0.9) for atk in ATTACKS for s in ARMED_SEEDS]
        r = p.map(armed_job, jobs, chunksize=5)
        res['armed'] = {atk: r[i * len(ARMED_SEEDS):(i + 1) * len(ARMED_SEEDS)] for i, atk in enumerate(ATTACKS)}
        print('armed done', time.time() - t, flush=True)
        res['fa'] = p.map(fa_job, FA_SEEDS)
        print('fa done', time.time() - t, flush=True)
        # sensitivity: motion-sensor detection probability (no-attack theft)
        sens = {}
        for pd in (0.7, 0.99):
            r = p.map(armed_job, [(s, 'none', pd) for s in ARMED_SEEDS], chunksize=5)
            sens[str(pd)] = r
        res['sens_pd'] = sens
    pickle.dump(res, open('results.pkl', 'wb'))
    print('all done', time.time() - t)

In [ ]:
%%writefile realdata.py
"""Turn the Porto taxi (ECML/PKDD 2015) trips into 1 Hz routes for the CATS-GSM simulator.

Real: the paths and speeds of the trips (15 s GPS samples).  Interpolated: everything between samples.
Modelled (as in the synthetic study): IMU, GPS noise/jamming/spoofing, serving cell / timing advance, energy.
"""
import json, glob, os
import numpy as np, pandas as pd
from scipy.interpolate import CubicSpline

LAT0, LON0 = 41.1579, -8.6291      # Porto centre
R_EARTH_X = 111320.0 * np.cos(np.radians(LAT0))
R_EARTH_Y = 110574.0


def find_csv(root='/kaggle/input'):
    cands = [p for p in glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
             if 'train' in os.path.basename(p).lower()] or glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
    if not cands: raise FileNotFoundError('No CSV found under ' + root)
    return max(cands, key=os.path.getsize)


def _route_from_points(pts, dt=15.0):
    """pts: (m,2) lon/lat at 15 s.  Returns x,y,v,th,a,w at 1 Hz (metres, m/s, rad) or None if the trip is unusable."""
    pts = np.asarray(pts, float)
    x = (pts[:, 0] - LON0) * R_EARTH_X
    y = (pts[:, 1] - LAT0) * R_EARTH_Y
    step = np.hypot(np.diff(x), np.diff(y))
    if step.max() > 40.0 * dt:                       # > 40 m/s average over one sample: GPS jump
        return None
    if np.hypot(x[-1] - x[0], y[-1] - y[0]) < 800.0:  # barely moved
        return None
    tk = np.arange(len(x)) * dt
    t = np.arange(0.0, tk[-1] + 1e-9, 1.0)
    xs = CubicSpline(tk, x)(t); ys = CubicSpline(tk, y)(t)
    k = np.ones(5) / 5.0
    vx = np.convolve(np.gradient(xs), k, mode='same'); vy = np.convolve(np.gradient(ys), k, mode='same')
    v = np.minimum(np.hypot(vx, vy), 40.0)
    th = np.arctan2(vy, vx)
    # hold heading while (almost) stopped, then unwrap
    for i in range(1, len(th)):
        if v[i] < 1.0: th[i] = th[i - 1]
    th = np.unwrap(th)
    a = np.gradient(v)
    w = np.gradient(th)
    w[v < 1.0] = 0.0
    w = np.clip(w, -np.pi / 4, np.pi / 4)
    return xs, ys, v, th, a, w


def load_routes(csv_path, n_routes=400, min_s=600, max_s=1800, seed=0, nrows=None, radius_m=9000.0):
    df = pd.read_csv(csv_path, usecols=['MISSING_DATA', 'POLYLINE'], nrows=nrows, dtype=str)
    df = df[df['MISSING_DATA'].astype(str).str.lower() == 'false']
    min_pts = int(min_s / 15) + 1
    df = df[df['POLYLINE'].str.len() > min_pts * 18]          # cheap pre-filter before JSON parsing
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(df))
    routes = []
    for i in idx:
        try:
            pts = json.loads(df['POLYLINE'].iloc[i])
        except Exception:
            continue
        if len(pts) < min_pts: continue
        pts = pts[: int(max_s / 15) + 1]
        r = _route_from_points(pts)
        if r is None: continue
        if np.max(np.hypot(r[0], r[1])) > radius_m: continue
        routes.append(r)
        if len(routes) >= n_routes: break
    return routes


def to_world_frame(routes, margin=3000.0):
    """Shift all routes so the bounding box starts at `margin`; returns (routes, area)."""
    xmin = min(r[0].min() for r in routes); ymin = min(r[1].min() for r in routes)
    xmax = max(r[0].max() for r in routes); ymax = max(r[1].max() for r in routes)
    out = []
    for r in routes:
        out.append((r[0] - xmin + margin, r[1] - ymin + margin) + tuple(r[2:]))
    area = max(xmax - xmin, ymax - ymin) + 2 * margin
    return out, float(area)

In [ ]:
%%writefile exp_real.py
"""CATS-GSM experiments on real Porto taxi routes (trace-driven).  Mirrors exp.py; results -> results.pkl for analysis.py"""
import sys, time, pickle, os
import numpy as np
import multiprocessing as mp
Pool = mp.get_context("fork").Pool
import world, algos as A
import realdata as R
from exp import DRIVE_CFG, SCHEMES, ATTACKS, run_cfg

CSV = os.environ.get('PORTO_CSV') or R.find_csv()
N_DRIVE = int(os.environ.get('N_DRIVE', 100))
N_ARMED = int(os.environ.get('N_ARMED', 150))
N_FA = int(os.environ.get('N_FA', 40))
NROWS = int(os.environ['NROWS']) if os.environ.get('NROWS') else None
NPROC = int(os.environ.get('NPROC', os.cpu_count() or 2))

print('reading', CSV, flush=True)
RAW = R.load_routes(CSV, n_routes=N_DRIVE + N_ARMED + N_FA, nrows=NROWS)
print('usable routes:', len(RAW), flush=True)
assert len(RAW) >= N_DRIVE + N_ARMED + N_FA, 'not enough usable trips; raise NROWS or lower the counts'
ROUTES, AREA = R.to_world_frame(RAW)
world.set_world(AREA)
print('world area %.1f km, %d base stations' % (AREA / 1000, len(world.WORLD['bts'])), flush=True)
DR, AR, FR = ROUTES[:N_DRIVE], ROUTES[N_DRIVE:N_DRIVE + N_ARMED], ROUTES[N_DRIVE + N_ARMED:]


def drive_job(i):
    tr = world.driving_trace(1000 + i, route=DR[i])
    return {f'{nm}|{v}': run_cfg(tr, nm, v) for nm, _, v in DRIVE_CFG}


def armed_job(args):
    i, atk, pd = args
    tr = world.armed_trace(2000 + i, 900, 1200, atk, acc_pd=pd, route=AR[i])
    out = {}
    for sc in SCHEMES:
        d = A.armed_detect(tr, sc)
        th = [a for a in d['alarms'] if a[0] >= tr['t0']]
        alarm = th[0] if th else None
        ex, ey, untr = A.track_after_alarm(tr, sc, alarm)
        err = A.track_error(tr, ex, ey, tr['t0'], tr['n'])
        out[sc] = dict(delay=(alarm[0] - tr['t0']) if alarm else None, reason=alarm[1] if alarm else None,
                       mean=float(err.mean()), p95=float(np.percentile(err, 95)), untrusted=untr)
    bts = world.WORLD['bts']
    cx = bts[tr['serving'][tr['t0']:], 0]; cy = bts[tr['serving'][tr['t0']:], 1]
    e = np.hypot(cx - tr['x'][tr['t0']:], cy - tr['y'][tr['t0']:])
    out['D4-cellID'] = dict(delay=None, reason=None, mean=float(e.mean()), p95=float(np.percentile(e, 95)), untrusted=None)
    return out


def fa_job(i):
    px, py = float(FR[i][0][0]), float(FR[i][1][0])
    tr = world.armed_trace(9100 + i, 86400, 0, 'none', theft=False, park=(px, py))
    out = {}
    for sc in SCHEMES:
        d = A.armed_detect(tr, sc, rearm=True)
        out[sc] = dict(alarms=len(d['alarms']), energy_J=float(d['meter'].total(86400, d['extra'])),
                       fixes=d['meter'].n_fix + d['meter'].n_fail)
    return out


if __name__ == '__main__':
    t = time.time(); res = {}
    with Pool(NPROC) as p:
        res['drive'] = p.map(drive_job, range(len(DR)))
        print('drive done %.0fs' % (time.time() - t), flush=True)
        jobs = [(i, atk, 0.9) for atk in ATTACKS for i in range(len(AR))]
        r = p.map(armed_job, jobs, chunksize=4)
        res['armed'] = {atk: r[k * len(AR):(k + 1) * len(AR)] for k, atk in enumerate(ATTACKS)}
        print('armed done %.0fs' % (time.time() - t), flush=True)
        res['fa'] = p.map(fa_job, range(len(FR)))
        print('false-alarm runs done %.0fs' % (time.time() - t), flush=True)
        sens = {}
        for pd_ in (0.7, 0.99):
            sens[str(pd_)] = p.map(armed_job, [(i, 'none', pd_) for i in range(len(AR))], chunksize=4)
        res['sens_pd'] = sens
    pickle.dump(res, open('results.pkl', 'wb'))
    print('all done %.0fs' % (time.time() - t))

In [ ]:
%%writefile analysis.py
import pickle, json, numpy as np
from scipy import stats
rng = np.random.default_rng(0)
res = pickle.load(open('results.pkl', 'rb'))
out = {}

# ---------------- driving
keys = list(res['drive'][0].keys())
N = len(res['drive'])
def arr(k, f): return np.array([r[k][f] for r in res['drive']])
fam = {}
for k in keys:
    nm, v = k.split('|'); fam.setdefault(nm, []).append(k)
rows = []
for k in keys:
    e = arr(k, 'mean'); p = arr(k, 'power_mW')
    bs = [rng.choice(e, N).mean() for _ in range(1000)]
    rows.append(dict(cfg=k, err=float(e.mean()), err_lo=float(np.percentile(bs, 2.5)), err_hi=float(np.percentile(bs, 97.5)),
                     p95=float(arr(k, 'p95').mean()), power=float(p.mean()), msgs=float(arr(k, 'msgs').mean()),
                     life=float(arr(k, 'life_days').mean())))
out['drive_rows'] = rows

def curve(name, idx):
    ks = fam[name]
    P = np.array([arr(k, 'power_mW')[idx].mean() for k in ks]); E = np.array([arr(k, 'mean')[idx].mean() for k in ks])
    o = np.argsort(P); return P[o], E[o]
def err_at(name, Pstar, idx):
    P, E = curve(name, idx)
    if len(P) < 2 or Pstar < P.min() or Pstar > P.max(): return np.nan
    return float(np.exp(np.interp(np.log(Pstar), np.log(P), np.log(E))))
levels = [50, 60, 80, 100]
match = {}
for Pstar in levels:
    for nm in ['D1-periodic', 'D2-distance', 'D3-DR', 'D3-IMU', 'CATS-noIMU', 'CATS']:
        vals = []
        for b in range(500):
            idx = rng.integers(0, N, N); vals.append(err_at(nm, Pstar, idx))
        point = err_at(nm, Pstar, np.arange(N))
        vv = np.array(vals); vv = vv[~np.isnan(vv)]
        match[f'{nm}@{Pstar}'] = dict(err=point, lo=float(np.percentile(vv, 2.5)) if len(vv) else None, hi=float(np.percentile(vv, 97.5)) if len(vv) else None)
out['matched'] = match
# paired improvement CATS vs others at matched power, bootstrap
imp = {}
for Pstar in levels:
    for base in ['D1-periodic', 'D2-distance', 'D3-DR', 'D3-IMU', 'CATS-noIMU']:
        r = []
        for b in range(500):
            idx = rng.integers(0, N, N)
            c, o = err_at('CATS', Pstar, idx), err_at(base, Pstar, idx)
            if not (np.isnan(c) or np.isnan(o)): r.append(100 * (1 - c / o))
        c0, o0 = err_at('CATS', Pstar, np.arange(N)), err_at(base, Pstar, np.arange(N))
        if r and not (np.isnan(c0) or np.isnan(o0)):
            imp[f'{base}@{Pstar}'] = dict(pct=100 * (1 - c0 / o0), lo=float(np.percentile(r, 2.5)), hi=float(np.percentile(r, 97.5)))
out['improvement_pct'] = imp

# ---------------- armed
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return float(c - h), float(c + h)
armed = {}
for atk, lst in res['armed'].items():
    armed[atk] = {}
    for sc in lst[0].keys():
        dl = [r[sc]['delay'] for r in lst]
        det300 = sum(1 for d in dl if d is not None and d <= 300)
        dv = [d for d in dl if d is not None]
        me = np.array([r[sc]['mean'] for r in lst]); p95 = np.array([r[sc]['p95'] for r in lst])
        armed[atk][sc] = dict(det300=det300, n=len(lst), det_ci=wilson(det300, len(lst)),
                              med_delay=float(np.median(dv)) if dv else None, p90_delay=float(np.percentile(dv, 90)) if dv else None,
                              track_mean=float(me.mean()), track_median=float(np.median(me)), track_p95=float(p95.mean()),
                              reasons={x: sum(1 for r in lst if r[sc]['reason'] == x) for x in set(r[sc]['reason'] for r in lst if r[sc]['reason'])})
out['armed'] = armed
tests = {}
for atk, lst in res['armed'].items():
    for base in ['A1', 'A2', 'A3', 'D4-cellID']:
        a = np.array([r['CATS']['mean'] for r in lst]); b = np.array([r[base]['mean'] for r in lst])
        try:
            w = stats.wilcoxon(a, b); tests[f'{atk}:CATS_vs_{base}'] = dict(p=float(w.pvalue), med_diff=float(np.median(a - b)))
        except Exception as e:
            tests[f'{atk}:CATS_vs_{base}'] = dict(p=None)
out['tests_track'] = tests
# detection paired McNemar (exact binomial) CATS vs A2/A3
mc = {}
for atk, lst in res['armed'].items():
    for base in ['A1', 'A2', 'A3']:
        b01 = sum(1 for r in lst if (r['CATS']['delay'] is not None and r['CATS']['delay'] <= 300) and not (r[base]['delay'] is not None and r[base]['delay'] <= 300))
        b10 = sum(1 for r in lst if not (r['CATS']['delay'] is not None and r['CATS']['delay'] <= 300) and (r[base]['delay'] is not None and r[base]['delay'] <= 300))
        p = stats.binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 > 0 else 1.0
        mc[f'{atk}:CATS_vs_{base}'] = dict(cats_only=b01, base_only=b10, p=float(p))
out['mcnemar'] = mc
# FA
fa = {}
for sc in res['fa'][0].keys():
    al = np.array([r[sc]['alarms'] for r in res['fa']]); en = np.array([r[sc]['energy_J'] for r in res['fa']])
    fa[sc] = dict(alarms_per_day=float(al.mean()), traces_with_fa=int((al > 0).sum()), n=len(al), kJ_per_day=float(en.mean() / 1000),
                  life_days=float(26640 / (en.mean() / 86400) / 86400), fixes_per_day=float(np.mean([r[sc]['fixes'] for r in res['fa']])))
out['fa'] = fa
# sensitivity
sens = {}
for pd, lst in list(res['sens_pd'].items()) + [('0.9', res['armed']['none'])]:
    sens[pd] = {sc: dict(det300=sum(1 for r in lst if r[sc]['delay'] is not None and r[sc]['delay'] <= 300), n=len(lst),
                         med=float(np.median([r[sc]['delay'] for r in lst if r[sc]['delay'] is not None] or [np.nan])))
                for sc in ['A1', 'A2', 'A3', 'CATS']}
out['sens_pd'] = sens
json.dump(out, open('analysis.json', 'w'), indent=1, default=float)

# figures
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.figure(figsize=(6.4, 4.4))
sty = {'D1-periodic': ('s', '#888'), 'D2-distance': ('^', '#b07'), 'D3-DR': ('v', '#d80'), 'D3-IMU': ('D', '#07a'), 'CATS-noIMU': ('x', '#6a3'), 'CATS': ('o', '#c00'), 'D4-cellID': ('*', '#000')}
for nm, (mk, c) in sty.items():
    P, E = curve(nm, np.arange(N)); plt.plot(P, E, marker=mk, color=c, label=nm, lw=1.3, ms=6)
plt.xscale('log'); plt.yscale('log'); plt.xlabel('Average tracker power (mW)'); plt.ylabel('Mean server tracking error (m)')
plt.grid(True, which='both', alpha=.3); plt.legend(fontsize=8); plt.tight_layout(); plt.savefig('fig_pareto.png', dpi=200)
print(json.dumps({k: out[k] for k in ['matched', 'improvement_pct', 'fa', 'sens_pd']}, indent=1, default=float))

In [ ]:
# ---- run the experiments (driving + theft + false alarms) ----
!python exp_real.py

In [ ]:
# ---- analysis: tables + figure ----
!python analysis.py

In [ ]:
# ---- package results ----
import shutil, json
from IPython.display import Image, display
display(Image('fig_pareto.png'))
os.makedirs('results_real', exist_ok=True)
for f in ('analysis.json','fig_pareto.png','results.pkl'): shutil.copy(f, 'results_real/'+f)
shutil.make_archive('results_real', 'zip', 'results_real')
print('Download results_real.zip from the Output panel and send it to Claude.')